# SHOTpy Tutorial: Using SHOT from Python

This notebook provides a comprehensive guide to using **SHOTpy**, the Python interface for the **SHOT** (Supporting Hyperplane Optimization Toolkit) solver.

SHOT is a solver for mixed-integer nonlinear programming (MINLP) problems that uses a combination of polyhedral outer approximation, extended supporting hyperplanes, and other techniques.

## Table of Contents
1. [Getting Started](#getting-started)
2. [Creating Variables](#creating-variables)
3. [Linear Problems](#linear-problems)
4. [Mixed-Integer Problems](#mixed-integer-problems)
5. [Quadratic Problems](#quadratic-problems)
6. [Nonlinear Problems](#nonlinear-problems)
7. [Reading Problems from Files](#reading-from-files)
8. [Configuring Solver Settings](#solver-settings)
9. [Advanced Features](#advanced-features)
10. [Callbacks](#callbacks)


## 1. Getting Started {#getting-started}

First, import the SHOTpy module and create an environment.

In [ ]:
import sys
import os

# Add current directory to Python path to find SHOTpy module
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd())

import SHOTpy

# Create a SHOT solver and get its environment
solver = SHOTpy.Solver()
env = solver.getEnvironment()
print("SHOT solver and environment created successfully!")

## 2. Creating Variables {#creating-variables}

Variables are the fundamental building blocks of optimization problems. SHOTpy supports:
- **Real** (continuous) variables
- **Integer** variables
- **Binary** variables

In [ ]:
# Create a continuous variable: x ∈ [0, 10]
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)

# Create an integer variable: y ∈ {0, 1, 2, ..., 5}
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Integer, 0.0, 5.0)

# Create a binary variable: z ∈ {0, 1}
z = SHOTpy.Variable("z", 2, SHOTpy.VariableType.Binary, 0.0, 1.0)

print(f"Created variables: {x.name}, {y.name}, {z.name}")

## 3. Linear Problems {#linear-problems}

Let's solve a simple linear program (LP):

$$
\begin{align*}
\text{minimize} \quad & 2x + 3y \\
\text{subject to} \quad & x + y \geq 1 \\
& x, y \geq 0
\end{align*}
$$

In [ ]:
# Create solver and get environment
solver = SHOTpy.Solver()
env = solver.getEnvironment()

# Create a new problem
problem = SHOTpy.Problem(env)

# Add variables
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 100.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.0, 100.0)
problem.addVariable(x)
problem.addVariable(y)

# Create linear objective: minimize 2x + 3y
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(2.0, x))
objective.add(SHOTpy.LinearTerm(3.0, y))
problem.setObjective(objective)

# Add constraint: x + y >= 1
constraint = SHOTpy.LinearConstraint(0, "c1", 1.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

# Finalize and solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

# Get results
sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

## 4. Mixed-Integer Problems {#mixed-integer-problems}

SHOT excels at solving mixed-integer problems. Here's a simple MIP:

$$
\begin{align*}
\text{maximize} \quad & 3x + 2y \\
\text{subject to} \quad & 2x + y \leq 10 \\
& x, y \in \mathbb{Z}_+
\end{align*}
$$

In [ ]:
# Create a MIP problem
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

# Add integer variables
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Integer, 0.0, 100.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Integer, 0.0, 100.0)
problem.addVariable(x)
problem.addVariable(y)

# Objective: maximize 3x + 2y
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Maximize)
objective.add(SHOTpy.LinearTerm(3.0, x))
objective.add(SHOTpy.LinearTerm(2.0, y))
problem.setObjective(objective)

# Constraint: 2x + y <= 10
constraint = SHOTpy.LinearConstraint(0, "budget", -SHOTpy.SHOT_DBL_MAX, 10.0)
constraint.add(SHOTpy.LinearTerm(2.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

# Solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]}, y = {sol.point[1]}")

## 5. Quadratic Problems {#quadratic-problems}

SHOT can handle quadratic objectives and constraints (QCQP):

$$
\begin{align*}
\text{minimize} \quad & x^2 + y^2 \\
\text{subject to} \quad & x + y \geq 2 \\
& x, y \geq 0
\end{align*}
$$

In [ ]:
# Create problem
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

# Variables
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Quadratic objective: x^2 + y^2
objective = SHOTpy.QuadraticObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.QuadraticTerm(1.0, x, x))  # x^2
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))  # y^2
problem.setObjective(objective)

# Linear constraint: x + y >= 2
constraint = SHOTpy.LinearConstraint(0, "sum_constraint", 2.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

# Solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

### Quadratic Constraints

You can also add quadratic constraints, such as circles or ellipses:

In [ ]:
# Problem: minimize x + y subject to x^2 + y^2 <= 4 (inside a circle)
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, -10.0, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, -10.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Linear objective
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
objective.add(SHOTpy.LinearTerm(1.0, y))
problem.setObjective(objective)

# Quadratic constraint: x^2 + y^2 <= 4
constraint = SHOTpy.QuadraticConstraint(0, "circle", -SHOTpy.SHOT_DBL_MAX, 4.0)
constraint.add(SHOTpy.QuadraticTerm(1.0, x, x))
constraint.add(SHOTpy.QuadraticTerm(1.0, y, y))
problem.addConstraint(constraint)

# Solve
problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")
print(f"Distance from origin: {(sol.point[0]**2 + sol.point[1]**2)**0.5:.4f}")

## 6. Nonlinear Problems {#nonlinear-problems}

SHOT is designed for nonlinear problems. You can use various nonlinear expressions:
- Exponential and logarithm: `SHOTpy.exp()`, `SHOTpy.log()`
- Trigonometric: `SHOTpy.sin()`, `SHOTpy.cos()`, `SHOTpy.tan()`
- Square root: `SHOTpy.sqrt()`
- Power functions: `x**n`
- Arithmetic operators: `+`, `-`, `*`, `/`

### Example: Geometric Programming

$$
\begin{align*}
\text{minimize} \quad & e^x + e^y \\
\text{subject to} \quad & e^{x+y} \leq 10 \\
& x, y \geq 0.1
\end{align*}
$$

In [ ]:
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.1, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Nonlinear objective: exp(x) + exp(y)
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.exp(x) + SHOTpy.exp(y))
problem.setObjective(objective)

# Nonlinear constraint: exp(x + y) <= 10
constraint = SHOTpy.NonlinearConstraint(0, "nl_constraint", -SHOTpy.SHOT_DBL_MAX, 10.0)
constraint.add(SHOTpy.exp(x + y))
problem.addConstraint(constraint)

problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

### Example: Mixed Terms

You can combine linear, quadratic, and nonlinear terms:

In [ ]:
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 5.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.1, 5.0)
problem.addVariable(x)
problem.addVariable(y)

# Mixed objective: x + y^2 + log(x)
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))           # Linear term
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))     # Quadratic term
objective.add(SHOTpy.log(x))                        # Nonlinear term
problem.setObjective(objective)

# Constraint: x + y >= 1
constraint = SHOTpy.LinearConstraint(0, "bound", 1.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

sol = solver.getPrimalSolution()
print(f"Optimal objective value: {sol.objValue:.4f}")
print(f"Optimal solution: x = {sol.point[0]:.4f}, y = {sol.point[1]:.4f}")

### Advanced Nonlinear Expressions

Complex nested expressions are also supported:

In [ ]:
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 3.14)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 1.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Complex expression: exp(x) * sin(x) + log(y) * cos(x)
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
expr = SHOTpy.exp(x) * SHOTpy.sin(x) + SHOTpy.log(y) * SHOTpy.cos(x)
objective.add(expr)
problem.setObjective(objective)

# Simple constraint
constraint = SHOTpy.LinearConstraint(0, "bound", 0.5, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
problem.addConstraint(constraint)

problem.finalize()
print("\nProblem structure:")
print(problem.toString())

## 7. Reading Problems from Files {#reading-from-files}

SHOT can read optimization problems from various file formats:
- **OSiL** (Optimization Services Instance Language)
- **GAMS** files (.gms)
- Other formats supported by modeling systems

In [ ]:
# Example: Reading from an OSiL file
# Note: Replace the path with an actual OSiL file path

# problem = SHOTpy.Problem(env)
# problem_file = "/path/to/problem.osil"
# 
# # Read the problem from file
# if problem.importModel(problem_file):
#     print(f"Successfully loaded problem from {problem_file}")
#     problem.finalize()
#     
#     solver = SHOTpy.Solver(env, problem)
#     solver.solveProblem()
#     
#     print(f"Objective value: {solver.getPrimalObjectiveValue():.4f}")
# else:
#     print("Failed to load problem file")

print("See commented code above for file loading example")

## 8. Configuring Solver Settings {#solver-settings}

SHOT provides extensive configuration options through its settings system.

### Accessing and Modifying Settings

In [ ]:
# Create solver and environment first
solver = SHOTpy.Solver()
env = solver.getEnvironment()

# Create a simple problem
problem = SHOTpy.Problem(env)
x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)

objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
problem.setObjective(objective)

problem.finalize()
solver.setProblem(problem)

# Get current values
print("Current Settings:")
time_limit = solver.getDoubleSetting("Termination.TimeLimit")
print(f"  Time limit: {time_limit} seconds")

abs_gap = solver.getDoubleSetting("Termination.ObjectiveGap.Absolute")
print(f"  Absolute gap tolerance: {abs_gap}")

# Modify settings
print("\nModifying settings...")
solver.updateSetting("Termination.TimeLimit", 300.0)  # 5 minutes
solver.updateSetting("Termination.ObjectiveGap.Absolute", 1e-6)

# Verify changes
new_time = solver.getDoubleSetting("Termination.TimeLimit")
new_gap = solver.getDoubleSetting("Termination.ObjectiveGap.Absolute")
print(f"  New time limit: {new_time} seconds")
print(f"  New absolute gap: {new_gap}")

### Common Settings

Here are some frequently used settings:

In [ ]:
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
problem.setObjective(objective)
problem.finalize()

solver.setProblem(problem)

# Termination criteria
solver.updateSetting("Termination.TimeLimit", 600.0)              # Max time in seconds
solver.updateSetting("Termination.ObjectiveGap.Absolute", 1e-5)   # Absolute gap
solver.updateSetting("Termination.ObjectiveGap.Relative", 1e-4)   # Relative gap

# Output settings
solver.updateSetting("Output.Console.LogLevel", 1)                # 0=off, 1=summary, 2=detailed

# MIP solver selection (convert enum to int)
solver.updateSetting("Dual.MIP.Solver", int(SHOTpy.MIPSolver.Gurobi))  # or Cplex, Cbc, Highs

# NLP solver selection
solver.updateSetting("Primal.FixedInteger.Solver", int(SHOTpy.PrimalNLPSolver.Ipopt))  # or GAMS

print("Settings configured successfully!")

# Verify some settings
print(f"Time limit: {solver.getDoubleSetting('Termination.TimeLimit')} seconds")
print(f"Log level: {solver.getIntSetting('Output.Console.LogLevel')}")

## 9. Advanced Features {#advanced-features}

### Monomial and Signomial Terms

For geometric programming, you can use signomial terms:

In [ ]:
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.1, 10.0)
problem.addVariable(x)
problem.addVariable(y)

# Create a signomial element: x^2 * y^(-1)
element1 = SHOTpy.SignomialElement(x, 2.0)
element2 = SHOTpy.SignomialElement(y, -1.0)

# Create signomial term: 3.0 * x^2 * y^(-1) - coefficient first, then elements
signomial = SHOTpy.SignomialTerm(3.0, [element1, element2])

# Alternative: Create monomial using variable-power pairs directly
monomial = SHOTpy.SignomialTerm(2.5, [(x, 2.0), (y, 3.0)])

# Use in constraint
constraint = SHOTpy.NonlinearConstraint(0, "signomial_constraint", -SHOTpy.SHOT_DBL_MAX, 100.0)
constraint.add(signomial)
problem.addConstraint(constraint)

# Simple objective
objective = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(1.0, x))
problem.setObjective(objective)

problem.finalize()
print("Problem with signomial terms created")
print(problem.toString())

### Examining Solution Details

In [ ]:
# Solve a simple problem
solver = SHOTpy.Solver()
env = solver.getEnvironment()
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Real, 0.0, 10.0)
problem.addVariable(x)
problem.addVariable(y)

objective = SHOTpy.QuadraticObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.QuadraticTerm(1.0, x, x))
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))
problem.setObjective(objective)

constraint = SHOTpy.LinearConstraint(0, "sum", 3.0, SHOTpy.SHOT_DBL_MAX)
constraint.add(SHOTpy.LinearTerm(1.0, x))
constraint.add(SHOTpy.LinearTerm(1.0, y))
problem.addConstraint(constraint)

problem.finalize()
solver.setProblem(problem)
solver.solveProblem()

# Get detailed results
print("Solution Details:")
sol = solver.getPrimalSolution()
print(f"  Primal objective: {sol.objValue:.6f}")
print(f"  Dual bound: {solver.getCurrentDualBound():.6f}")

solutions = solver.getPrimalSolutions()
print(f"  Number of solutions found: {len(solutions)}")

if len(solutions) > 0:
    print(f"  Best solution: x = {solutions[0].point[0]:.4f}, y = {solutions[0].point[1]:.4f}")

# Get problem name (if set)
print(f"  Problem name: {problem.name if hasattr(problem, 'name') else 'N/A'}")

### Checking Problem Properties

In [ ]:
# Create a problem with various features
problem = SHOTpy.Problem(env)

x = SHOTpy.Variable("x", 0, SHOTpy.VariableType.Real, 0.1, 10.0)
y = SHOTpy.Variable("y", 1, SHOTpy.VariableType.Integer, 0.0, 5.0)
problem.addVariable(x)
problem.addVariable(y)

# Mix of linear, quadratic, and nonlinear terms
objective = SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
objective.add(SHOTpy.LinearTerm(2.0, x))
objective.add(SHOTpy.QuadraticTerm(1.0, y, y))
objective.add(SHOTpy.log(x))
problem.setObjective(objective)

constraint1 = SHOTpy.LinearConstraint(0, "linear", 1.0, SHOTpy.SHOT_DBL_MAX)
constraint1.add(SHOTpy.LinearTerm(1.0, x))
problem.addConstraint(constraint1)

constraint2 = SHOTpy.NonlinearConstraint(1, "nonlinear", -SHOTpy.SHOT_DBL_MAX, 10.0)
constraint2.add(SHOTpy.exp(x))
problem.addConstraint(constraint2)

problem.finalize()

# Display problem structure with indicators
print("Problem Structure:")
print(problem.toString())
print("\nIndicators legend:")
print("  L = Linear terms")
print("  Q = Quadratic terms")
print("  S = Signomial terms (power functions)")
print("  M = Monomial terms")
print("  E = General nonlinear expressions")

## 10. Callbacks {#callbacks}

SHOTpy exposes SHOT's callback system via `solver.registerCallback(event, fn)`. Six event types are available through `SHOTpy.EventType`:

| Event type | Callback signature | Description |
|---|---|---|
| `NewPrimalSolution` | `fn(data) -> None` | Called every time a new primal (incumbent) solution is found |
| `PrimalSolutionCandidateSelection` | `fn(data) -> None` | Called when a primal candidate is being evaluated |
| `UserTerminationCheck` | `fn(data) -> bool \| None` | Return `True` to stop the solver early; `None`/`False` continues |
| `ExternalDualBound` | `fn(data) -> float \| None` | Return a tighter dual bound, or `None` to skip |
| `ExternalPrimalSolution` | `fn(data) -> list[float] \| None` | Return a feasible primal point to inject, or `None` to skip |
| `ExternalHyperplaneSelection` | `fn(data) -> list[ExternalHyperplane] \| None` | Supply custom supporting hyperplane cuts |

Each callback receives a typed data object (e.g. `PrimalSolutionCallbackData`, `TerminationCallbackData`) with fields like `iterationNumber`, `currentDualBound`, `relativeGap`, and `solutionStatistics`.


In [ ]:
import SHOTpy

# ── Build the ex1223b MINLP ────────────────────────────────────────────────
solver = SHOTpy.Solver()
env    = solver.getEnvironment()

problem = SHOTpy.Problem(env)
problem.name = "ex1223b_callbacks"

x1 = SHOTpy.Variable("x1", 0, SHOTpy.VariableType.Real,   0.0, 10.0)
x2 = SHOTpy.Variable("x2", 1, SHOTpy.VariableType.Real,   0.0, 10.0)
x3 = SHOTpy.Variable("x3", 2, SHOTpy.VariableType.Real,   0.0, 10.0)
b4 = SHOTpy.Variable("b4", 3, SHOTpy.VariableType.Binary, 0.0, 1.0)
b5 = SHOTpy.Variable("b5", 4, SHOTpy.VariableType.Binary, 0.0, 1.0)
b6 = SHOTpy.Variable("b6", 5, SHOTpy.VariableType.Binary, 0.0, 1.0)
b7 = SHOTpy.Variable("b7", 6, SHOTpy.VariableType.Binary, 0.0, 1.0)
for v in [x1, x2, x3, b4, b5, b6, b7]:
    problem.addVariable(v)

obj_expr = ((b4 - 1.0)**2 + (b5 - 2.0)**2 + (b6 - 1.0)**2
            - SHOTpy.log(1.0 + b7)
            + (x1 - 1.0)**2 + (x2 - 2.0)**2 + (x3 - 3.0)**2)
problem.setObjective(
    SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize,
                                      obj_expr, 0.0))

lt1 = SHOTpy.LinearTerms()
for v in [x1, x2, x3, b4, b5, b6]:
    lt1.add(SHOTpy.LinearTerm(1.0, v))
problem.addConstraint(SHOTpy.LinearConstraint(0, "e1", lt1, SHOTpy.SHOT_DBL_MIN, 5.0))

problem.addConstraint(SHOTpy.NonlinearConstraint(
    1, "e2", b6**2 + x1**2 + x2**2 + x3**2, SHOTpy.SHOT_DBL_MIN, 5.5))

for idx, (va, vb, rhs, name) in enumerate(
        [(x1, b4, 1.2, "e3"), (x2, b5, 1.8, "e4"),
         (x3, b6, 2.5, "e5"), (x1, b7, 1.2, "e6")], start=2):
    lt = SHOTpy.LinearTerms()
    lt.add(SHOTpy.LinearTerm(1.0, va)); lt.add(SHOTpy.LinearTerm(1.0, vb))
    problem.addConstraint(SHOTpy.LinearConstraint(idx, name, lt, SHOTpy.SHOT_DBL_MIN, rhs))

problem.addConstraint(SHOTpy.NonlinearConstraint(
    6, "e7", b5**2 + x2**2, SHOTpy.SHOT_DBL_MIN, 1.64))
problem.addConstraint(SHOTpy.NonlinearConstraint(
    7, "e8", b6**2 + x3**2, SHOTpy.SHOT_DBL_MIN, 4.25))
problem.addConstraint(SHOTpy.NonlinearConstraint(
    8, "e9", b5**2 + x3**2, SHOTpy.SHOT_DBL_MIN, 4.64))
problem.finalize()

# ── Example 1: track every new primal solution ────────────────────────────
primal_log = []

def on_new_primal(data):
    primal_log.append({
        "iter":      data.iterationNumber,
        "obj":       data.objectiveValue,
        "dual":      data.currentDualBound,
        "gap (%)":   round(data.relativeGap * 100, 4),
    })

solver.registerCallback(SHOTpy.EventType.NewPrimalSolution, on_new_primal)

# ── Example 2: stop after the relative gap is tight enough ───────────────
def early_stop(data):
    # Terminate once the gap drops below 1 %
    return data.relativeGap < 0.01

solver.registerCallback(SHOTpy.EventType.UserTerminationCheck, early_stop)

# ── Suppress console output and solve ────────────────────────────────────
solver.updateSetting("Output.Console.LogLevel", 6)  # 6 = Off
solver.setProblem(problem)
solver.solveProblem()

# ── Results ───────────────────────────────────────────────────────────────
print(f"Status : {solver.getModelReturnStatus()}")
print(f"Optimal: {solver.getPrimalBound():.6f}  (known optimum ≈ 4.5796)")
print(f"\nPrimal solution improvements ({len(primal_log)} updates):")
print(f"  {'iter':>5}  {'objective':>12}  {'dual bound':>12}  {'gap %':>8}")
for row in primal_log:
    print(f"  {row['iter']:>5}  {row['obj']:>12.6f}  {row['dual']:>12.6f}  {row['gap (%)']:>8.4f}")

# ── Example 3: inspect an ExternalHyperplane object ──────────────────────
hp = SHOTpy.ExternalHyperplane()
hp.variableIndexes      = [0, 2]        # x1 and x3
hp.variableCoefficients = [1.5, -0.5]   # 1.5*x1 - 0.5*x3 <= rhs
hp.rhsValue             = 3.0
hp.isGlobal             = True
hp.source               = SHOTpy.HyperplaneSource.External
hp.description          = "my_custom_cut"

print("\nExample ExternalHyperplane:")
print(f"  indexes : {hp.variableIndexes}")
print(f"  coeffs  : {hp.variableCoefficients}")
print(f"  rhs     : {hp.rhsValue}")
print(f"  global  : {hp.isGlobal}")
print(f"  source  : {hp.source}")

### `PrimalSolutionCandidateSelection` callback

`PrimalSolutionCandidateSelection` fires **before** the solver decides whether to accept a
candidate as the new incumbent — unlike `NewPrimalSolution`, which fires only after acceptance.
Both share the same `PrimalSolutionCallbackData` type:

| Field | Type | Description |
|---|---|---|
| `iterationNumber` | `int` | Current iteration |
| `objectiveValue` | `float` | Objective value of the candidate point |
| `currentDualBound` | `float` | Dual bound at this point in the search |
| `relativeGap` | `float` | Relative optimality gap |
| `isMinimization` | `bool` | `True` for minimization problems |
| `solution` | `list[float]` | Candidate variable values |

**Return value:** `True` or `None` to let the solver check the candidate normally;
`False` to reject it outright (the feasibility check is skipped and the candidate
is never recorded as a primal solution).

This callback is useful for:
- *Logging* all candidate points examined by the primal heuristics (return `None`/`True`)
- *Filtering* candidates based on problem-specific criteria — e.g. enforcing extra
  constraints that are not in the model (return `False` to reject)

In [ ]:
import SHOTpy

# ── Build the ex1223b MINLP (same problem used in the main callback example) ──
def _build():
    s = SHOTpy.Solver()
    s.updateSetting("Output.Console.LogLevel", 6)  # silent
    e = s.getEnvironment()
    p = SHOTpy.Problem(e); p.name = "ex1223b_cand"
    x1 = SHOTpy.Variable("x1", 0, SHOTpy.VariableType.Real,   0.0, 10.0)
    x2 = SHOTpy.Variable("x2", 1, SHOTpy.VariableType.Real,   0.0, 10.0)
    x3 = SHOTpy.Variable("x3", 2, SHOTpy.VariableType.Real,   0.0, 10.0)
    b4 = SHOTpy.Variable("b4", 3, SHOTpy.VariableType.Binary, 0.0, 1.0)
    b5 = SHOTpy.Variable("b5", 4, SHOTpy.VariableType.Binary, 0.0, 1.0)
    b6 = SHOTpy.Variable("b6", 5, SHOTpy.VariableType.Binary, 0.0, 1.0)
    b7 = SHOTpy.Variable("b7", 6, SHOTpy.VariableType.Binary, 0.0, 1.0)
    for v in [x1, x2, x3, b4, b5, b6, b7]: p.addVariable(v)
    obj = ((b4-1)**2+(b5-2)**2+(b6-1)**2-SHOTpy.log(1+b7)+(x1-1)**2+(x2-2)**2+(x3-3)**2)
    p.setObjective(SHOTpy.NonlinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize, obj, 0.0))
    lt1 = SHOTpy.LinearTerms()
    for v in [x1,x2,x3,b4,b5,b6]: lt1.add(SHOTpy.LinearTerm(1.0, v))
    p.addConstraint(SHOTpy.LinearConstraint(0, "e1", lt1, SHOTpy.SHOT_DBL_MIN, 5.0))
    p.addConstraint(SHOTpy.NonlinearConstraint(1,"e2",b6**2+x1**2+x2**2+x3**2,SHOTpy.SHOT_DBL_MIN,5.5))
    for idx,(va,vb,rhs,nm) in enumerate([(x1,b4,1.2,"e3"),(x2,b5,1.8,"e4"),(x3,b6,2.5,"e5"),(x1,b7,1.2,"e6")],start=2):
        lt=SHOTpy.LinearTerms(); lt.add(SHOTpy.LinearTerm(1.0,va)); lt.add(SHOTpy.LinearTerm(1.0,vb))
        p.addConstraint(SHOTpy.LinearConstraint(idx,nm,lt,SHOTpy.SHOT_DBL_MIN,rhs))
    p.addConstraint(SHOTpy.NonlinearConstraint(6,"e7",b5**2+x2**2,SHOTpy.SHOT_DBL_MIN,1.64))
    p.addConstraint(SHOTpy.NonlinearConstraint(7,"e8",b6**2+x3**2,SHOTpy.SHOT_DBL_MIN,4.25))
    p.addConstraint(SHOTpy.NonlinearConstraint(8,"e9",b5**2+x3**2,SHOTpy.SHOT_DBL_MIN,4.64))
    p.finalize(); s.setProblem(p)
    return s

# ── Run 1: accept all candidates (return None = accept) ───────────────────────
s1 = _build()
cand1, acc1 = [], []
s1.registerCallback(SHOTpy.EventType.PrimalSolutionCandidateSelection,
                    lambda d: cand1.append(round(d.objectiveValue, 4)) or None)
s1.registerCallback(SHOTpy.EventType.NewPrimalSolution,
                    lambda d: acc1.append(round(d.objectiveValue, 4)))
s1.solveProblem()

# ── Run 2: reject sub-optimal candidates (obj > 5) ────────────────────────────
s2 = _build()
cand2, acc2 = [], []

def on_candidate(data):
    cand2.append(round(data.objectiveValue, 4))
    # Return False to skip feasibility checking for this candidate
    return data.objectiveValue <= 5.0

s2.registerCallback(SHOTpy.EventType.PrimalSolutionCandidateSelection, on_candidate)
s2.registerCallback(SHOTpy.EventType.NewPrimalSolution,
                    lambda d: acc2.append(round(d.objectiveValue, 4)))
s2.solveProblem()

# ── Compare results ────────────────────────────────────────────────────────────
print("── Accept all ────────────────────────────────────────────────")
print(f"  Candidates seen   : {cand1}")
print(f"  Incumbents found  : {acc1}")
print()
print("── Reject obj > 5 ────────────────────────────────────────────")
print(f"  Candidates seen   : {cand2}")
print(f"  Incumbents found  : {acc2}")
print(f"  (rejected {sum(1 for v in cand2 if v > 5.0)} candidate(s) with obj > 5)")

### External Hyperplane Callback: Solving `shot_ex_jogo`

This example demonstrates the `ExternalHyperplaneSelection` callback by solving a small MINLP
whose problem data comes from `test/data/shot_ex_jogo.gms`.  The problem is constructed
directly through the Python API.

$$
\min_{x_1,\,x_2} \; -x_1 - x_2
$$
subject to

$$
2x_1 - 3x_2 \le 2, \qquad x_1 \in [1,20],\; x_2 \in \mathbb{Z} \cap [1,20]
$$
$$
0.15(x_1-8)^2 + 0.1(x_2-6)^2 + \frac{0.025\,e^{x_1}}{x_2^2} \le 5
$$
$$
\frac{1}{x_1} + \frac{1}{x_2} - \sqrt{x_1}\,\sqrt{x_2} \le -4
$$

All built-in SHOT hyperplane generation is disabled (`CutStrategy = OnlyExternal`).
The callback computes a linearisation cut (supporting hyperplane) for the most-violated
nonlinear constraint at every MIP node and returns it to SHOT as an `ExternalHyperplane`.


In [ ]:
import SHOTpy

# ── Create solver and configure settings ─────────────────────────────────────
solver = SHOTpy.Solver()
env    = solver.getEnvironment()

solver.updateSetting("Output.Console.LogLevel", 2)

# Disable all built-in hyperplane generation; cuts come exclusively from the callback
solver.updateSetting("Model.Convexity.AssumeConvex", True)   # True\n
solver.updateSetting("Model.Reformulation.Constraint.PartitionQuadraticTerms", 2)  # ES_PartitionNonlinearSums::Never
solver.updateSetting("Dual.CutStrategy", 2)    # ES_HyperplaneCutStrategy::OnlyExternal
solver.updateSetting("Dual.TreeStrategy", 0)    # ES_TreeStrategy::MultiTree

# ── Build the shot_ex_jogo problem via the Python API ─────────────────────────
#
#   min  -x1 - x2
#   s.t. 2*x1 - 3*x2                            <=  2   (l,  linear)
#        0.15*(x1-8)^2 + 0.1*(x2-6)^2
#          + 0.025*exp(x1)/x2^2                 <=  5   (c1, nonlinear)
#        1/x1 + 1/x2 - sqrt(x1)*sqrt(x2)        <= -4   (c2, nonlinear / signomial)
#   x1 in [1, 20]  (continuous)
#   x2 in [1, 20]  (integer)

problem      = SHOTpy.Problem(env)
problem.name = "shot_ex_jogo"

x1 = SHOTpy.Variable("x1", 0, SHOTpy.VariableType.Real,    1.0, 20.0)
x2 = SHOTpy.Variable("x2", 1, SHOTpy.VariableType.Integer, 1.0, 20.0)
problem.addVariable(x1)
problem.addVariable(x2)

# Objective: minimize -x1 - x2
obj = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
obj.add(SHOTpy.LinearTerm(-1.0, x1))
obj.add(SHOTpy.LinearTerm(-1.0, x2))
problem.setObjective(obj)

# Constraint l: 2*x1 - 3*x2 <= 2
lt_l = SHOTpy.LinearTerms()
lt_l.add(SHOTpy.LinearTerm( 2.0, x1))
lt_l.add(SHOTpy.LinearTerm(-3.0, x2))
problem.addConstraint(SHOTpy.LinearConstraint(0, "l", lt_l, SHOTpy.SHOT_DBL_MIN, 2.0))

# Constraint c1: 0.15*(x1-8)^2 + 0.1*(x2-6)^2 + 0.025*exp(x1)/x2^2 <= 5
c1 = SHOTpy.NonlinearConstraint(1, "c1", SHOTpy.SHOT_DBL_MIN, 5.0)
c1.add(0.15 * (x1 - 8.0)**2 + 0.1 * (x2 - 6.0)**2 + 0.025 * SHOTpy.exp(x1) / x2**2)
problem.addConstraint(c1)

# Constraint c2: 1/x1 + 1/x2 - x1^0.5 * x2^0.5 <= -4
# Expressed as signomial terms: x1^(-1) + x2^(-1) - x1^0.5 * x2^0.5
c2 = SHOTpy.NonlinearConstraint(2, "c2", SHOTpy.SHOT_DBL_MIN, -4.0)
c2.add(SHOTpy.SignomialTerm( 1.0, [(x1, -1.0)]))              # 1/x1
c2.add(SHOTpy.SignomialTerm( 1.0, [(x2, -1.0)]))              # 1/x2
c2.add(SHOTpy.SignomialTerm(-1.0, [(x1, 0.5), (x2, 0.5)]))   # -sqrt(x1)*sqrt(x2)
problem.addConstraint(c2)

# Keep a reference to the nonlinear constraints for gradient computations in the callback
nonlinear_constraints = [c1, c2]

problem.finalize()

# Pass the same problem as both original and reformulated to bypass internal reformulation
solver.setProblem(problem, problem)

print("Problem created:")
print(problem.toString())

# ── Register external hyperplane callback ─────────────────────────────────────
#
# At every MIP node SHOT calls this function with a list of solution points.
# For each point we linearise the most-violated nonlinear constraint at that
# point and return the resulting cut as an ExternalHyperplane.
#
# Supporting hyperplane for  g(x) <= b  at  x0:
#   grad g(x0) . x  <=  grad g(x0) . x0  -  (g(x0) - b)
#                   <=  sum_i grad_i * x0_i  -  violation

cut_log = []   # record (iteration, constraint name, violation) for display

def generate_hyperplanes(data):
    hyperplanes = []

    if not data.solutionPoints or data.iterationNumber == 0:
        return hyperplanes

    reform_problem = data.reformulatedProblem

    for sol_point in data.solutionPoints:
        dev_idx = sol_point.maxDeviation.index

        # Index must be valid inside the nonlinear-constraint list
        if dev_idx < 0 or dev_idx >= len(reform_problem.nonlinearConstraints):
            continue

        violation = sol_point.maxDeviation.value
        if violation <= 0.0:
            continue   # constraint is satisfied at this point

        # Looks up by the constraint's own .index attribute
        constraint = next(
            (nlc for nlc in reform_problem.nonlinearConstraints if nlc.index == dev_idx), None)
        
        point      = sol_point.point

        # Sparse gradient dict {var_index: coefficient}
        gradient = constraint.calculateGradient(point)

        if not gradient:
            continue

        var_indices  = list(gradient.keys())
        var_coeffs   = list(gradient.values())

        # rhs = grad . x0 - violation
        rhs = sum(gradient[i] * point[i] for i in var_indices) - violation

        hp                      = SHOTpy.ExternalHyperplane()
        hp.variableIndexes      = var_indices
        hp.variableCoefficients = var_coeffs
        hp.rhsValue             = rhs
        hp.isGlobal             = True
        hp.source               = SHOTpy.HyperplaneSource.External
        hp.description          = f"ext_hp_iter{data.iterationNumber}_c{dev_idx}"

        hyperplanes.append(hp)
        cut_log.append((data.iterationNumber, constraint.name, round(violation, 6)))

        break   # one cut per callback invocation is sufficient

    return hyperplanes

solver.registerCallback(SHOTpy.EventType.ExternalHyperplaneSelection, generate_hyperplanes)

# ── Solve ──────────────────────────────────────────────────────────────────────
print("\nSolving with external hyperplanes only...\n")
solver.solveProblem()

# ── Results ───────────────────────────────────────────────────────────────────
print(f"\nStatus : {solver.getModelReturnStatus()}")

if solver.getPrimalSolutions():
    sol = solver.getPrimalSolution()
    print(f"Objective value : {sol.objValue:.6f}")
    print(f"x1 = {sol.point[0]:.6f}  (continuous)")
    print(f"x2 = {sol.point[1]:.6f}  (integer)")
else:
    print("No primal solution found.")

print(f"\nExternal cuts added: {len(cut_log)}")
if cut_log:
    print(f"  {'iter':>6}  {'constraint':>12}  {'violation':>12}")
    for (it, cname, viol) in cut_log[:10]:
        print(f"  {it:>6}  {cname:>12}  {viol:>12.6f}")
    if len(cut_log) > 10:
        print(f"  ... ({len(cut_log) - 10} more)")

### External Dual Bound and Primal Solution Callbacks

SHOT exposes two additional callbacks that let the user feed external information
directly into the branch-and-bound search:

| Callback | Signature | Effect |
|---|---|---|
| `ExternalDualBound` | `fn(DualBoundCallbackData) -> float \| None` | Tighten the dual (lower) bound. Return `None` to skip. |
| `ExternalPrimalSolution` | `fn(ExternalPrimalSolutionCallbackData) -> list[float] \| None` | Inject a candidate primal point. Return `None` to skip. |

**`DualBoundCallbackData`** fields:
- `iterationNumber` – current iteration
- `currentDualBound` – solver's current dual bound
- `currentPrimalBound` – solver's current primal bound

**`ExternalPrimalSolutionCallbackData`** fields:
- `iterationNumber`, `currentDualBound`, `currentPrimalBound`

The example below solves `shot_ex_jogo` again, this time with both callbacks active:

1. **`ExternalDualBound`** – waits until iteration 3 before injecting the
   known optimal dual bound `−20.9036`, tightening the gap early.
2. **`ExternalPrimalSolution`** – on the very first call provides the
   known optimal point `[8.903615, 12.0]` (previously verified by a warm-up
   solve), letting the solver skip most of the primal search.

Together they cause the solver to converge in a single main iteration.

In [ ]:
import SHOTpy
import math

# ── Warm-up solve to collect a verified feasible primal solution ──────────────
#
# We first solve the problem normally to get a solution point that SHOT has
# already checked for feasibility.  This avoids floating-point rounding issues
# when re-injecting the point in the main solve.

ws = SHOTpy.Solver()
ws.updateSetting("Output.Console.LogLevel", 6)   # silent
ws.updateSetting("Model.Convexity.AssumeConvex", True)
env_ws = ws.getEnvironment()

def build_shot_ex_jogo(env):
    problem = SHOTpy.Problem(env)
    problem.name = "shot_ex_jogo"

    x1 = SHOTpy.Variable("x1", 0, SHOTpy.VariableType.Real,    1.0, 20.0)
    x2 = SHOTpy.Variable("x2", 1, SHOTpy.VariableType.Integer, 1.0, 20.0)
    problem.addVariable(x1)
    problem.addVariable(x2)

    obj = SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize)
    obj.add(SHOTpy.LinearTerm(-1.0, x1))
    obj.add(SHOTpy.LinearTerm(-1.0, x2))
    problem.setObjective(obj)

    lt_l = SHOTpy.LinearTerms()
    lt_l.add(SHOTpy.LinearTerm( 2.0, x1))
    lt_l.add(SHOTpy.LinearTerm(-3.0, x2))
    problem.addConstraint(SHOTpy.LinearConstraint(0, "l", lt_l, SHOTpy.SHOT_DBL_MIN, 2.0))

    c1 = SHOTpy.NonlinearConstraint(1, "c1", SHOTpy.SHOT_DBL_MIN, 5.0)
    c1.add(0.15 * (x1 - 8.0)**2 + 0.1 * (x2 - 6.0)**2 + 0.025 * SHOTpy.exp(x1) / x2**2)
    problem.addConstraint(c1)

    c2 = SHOTpy.NonlinearConstraint(2, "c2", SHOTpy.SHOT_DBL_MIN, -4.0)
    c2.add(SHOTpy.SignomialTerm( 1.0, [(x1, -1.0)]))
    c2.add(SHOTpy.SignomialTerm( 1.0, [(x2, -1.0)]))
    c2.add(SHOTpy.SignomialTerm(-1.0, [(x1, 0.5), (x2, 0.5)]))
    problem.addConstraint(c2)

    problem.finalize()
    return problem

ws.setProblem(build_shot_ex_jogo(env_ws), build_shot_ex_jogo(env_ws))
ws.solveProblem()

known_point = list(ws.getPrimalSolution().point)   # verified-feasible optimal point
known_obj   = ws.getPrimalSolution().objValue
known_dual  = -20.903647306104258                  # from reference solution

print(f"Warm-up solve finished.")
print(f"  Optimal point : x1={known_point[0]:.6f}, x2={known_point[1]:.1f}")
print(f"  Optimal obj   : {known_obj:.6f}")
print(f"  Known dual    : {known_dual:.6f}")

# ── Main solve with ExternalDualBound and ExternalPrimalSolution callbacks ────

solver = SHOTpy.Solver()
solver.updateSetting("Output.Console.LogLevel", 2)
solver.updateSetting("Model.Convexity.AssumeConvex", True)
solver.updateSetting("Dual.Relaxation.Use", False)
solver.updateSetting("Dual.TreeStrategy", 0)    # MultiTree (required for ExternalDualBound)
env = solver.getEnvironment()
solver.setProblem(build_shot_ex_jogo(env), build_shot_ex_jogo(env))

# ── Callback state ────────────────────────────────────────────────────────────
INJECT_DUAL_AFTER_ITER = 3   # wait this many iterations before tightening the dual bound

dual_log    = []
primal_log  = []
event_log   = []

# ── ExternalDualBound callback ────────────────────────────────────────────────
#
# Called once per main iteration.  We wait until iteration > INJECT_DUAL_AFTER_ITER
# so that the solver has already made some progress on its own, then inject the
# known optimal dual bound to close the gap immediately.

dual_injected = [False]

def provide_dual_bound(data):
    entry = {
        "iter":    data.iterationNumber,
        "current": data.currentDualBound,
        "injected": None,
    }
    if (not dual_injected[0]
            and data.iterationNumber >= INJECT_DUAL_AFTER_ITER
            and data.currentDualBound < known_dual):
        dual_injected[0]  = True
        entry["injected"] = known_dual
        event_log.append(f"iter {data.iterationNumber:>2}: [ExternalDualBound]  "
                         f"current={data.currentDualBound:.4f}  "
                         f"→ injecting known dual {known_dual:.6f}")
        dual_log.append(entry)
        return known_dual

    event_log.append(f"iter {data.iterationNumber:>2}: [ExternalDualBound]  "
                     f"current={data.currentDualBound:.4f}  → (no update)")
    dual_log.append(entry)
    return None   # None means "no update from my side"

# ── ExternalPrimalSolution callback ──────────────────────────────────────────
#
# Called once per main iteration.  On the very first call we hand back the
# warm-up optimal point; SHOT validates it and, if feasible, accepts it as the
# new incumbent (triggering a NewPrimalSolution event).

primal_injected = [False]

def provide_primal_solution(data):
    entry = {
        "iter":    data.iterationNumber,
        "current": data.currentPrimalBound,
        "injected": None,
    }
    if not primal_injected[0]:
        primal_injected[0]  = True
        entry["injected"]   = known_point
        event_log.append(f"iter {data.iterationNumber:>2}: [ExternalPrimalSolution]  "
                         f"current={data.currentPrimalBound:.4f}  "
                         f"→ injecting known point {[round(v,4) for v in known_point]}")
        primal_log.append(entry)
        return known_point   # list[float] with one value per variable

    event_log.append(f"iter {data.iterationNumber:>2}: [ExternalPrimalSolution]  "
                     f"current={data.currentPrimalBound:.4f}  → (no update)")
    primal_log.append(entry)
    return None

# ── NewPrimalSolution notification ───────────────────────────────────────────
def on_new_primal(data):
    event_log.append(f"iter {data.iterationNumber:>2}: [NewPrimalSolution]  "
                     f"obj={data.objectiveValue:.6f}  "
                     f"point={[round(v,4) for v in data.solution]}")

solver.registerCallback(SHOTpy.EventType.NewPrimalSolution,      on_new_primal)
solver.registerCallback(SHOTpy.EventType.ExternalDualBound,      provide_dual_bound)
solver.registerCallback(SHOTpy.EventType.ExternalPrimalSolution, provide_primal_solution)

print(f"\nSolving with ExternalDualBound (inject after iter {INJECT_DUAL_AFTER_ITER}) "
      f"and ExternalPrimalSolution (inject on first call)...\n")
solver.solveProblem()

# ── Print event log ───────────────────────────────────────────────────────────
print("\nCallback events (in order):")
for msg in event_log:
    print(f"  {msg}")

# ── Final results ─────────────────────────────────────────────────────────────
print(f"\nStatus    : {solver.getModelReturnStatus()}")
if solver.getPrimalSolutions():
    sol = solver.getPrimalSolution()
    print(f"Objective : {sol.objValue:.6f}  (known optimum {known_obj:.6f})")
    print(f"x1 = {sol.point[0]:.6f}  x2 = {sol.point[1]:.1f}")
    print(f"Iterations: {solver.getSolutionStatistics().numberOfIterations}")
else:
    print("No primal solution found.")

---
### ESH Interior Point Callback: Injecting Custom Interior Points

The **`ExternalESHRootsearchPointsSelection`** event lets you control which interior points
are used by the Extended Supporting Hyperplane (ESH) algorithm.  The ESH method needs a
strictly interior point of the feasible region to generate supporting hyperplanes; normally
SHOT finds this point automatically by solving a cutting-plane minimax NLP.  The callback
gives you two complementary powers:

1. **Inspect / filter** the point(s) found by the internal strategy — the callback is called
   with `data.currentInteriorPoints` containing the points the solver just found.  You may
   return `None` or `[]` to leave them unchanged, or return a new list to replace them.

2. **Bypass the internal search entirely** by setting `ESH.InteriorPoint.Strategy = 1`
   ("Only external (through callback)").  The solver will skip the minimax NLP and rely
   completely on the points you return from the callback.  This is useful when you already
   know a feasible interior point — for example from a previous solve.

```
ESH.InteriorPoint.Strategy
  0  Use internal strategy (default) — callback fires after, allowing modification
  1  Only external (through callback) — internal NLP is skipped entirely
```

**Callback signature:**
```python
def on_interior_points(data: SHOTpy.ESHInteriorPointCallbackData
                       ) -> list[list[float]] | None:
    ...
```

| Field | Type | Description |
|---|---|---|
| `data.currentInteriorPoints` | `list[list[float]]` | Points found by the internal strategy (empty if `OnlyExternal`) |
| `data.originalProblem` | `Problem` | The original problem object |
| `data.reformulatedProblem` | `Problem` | The reformulated problem used internally by SHOT |
| `data.solutionStatistics` | `SolutionStatistics` | Current solver statistics |

The example below uses a **two-phase** pattern:

* **Phase 1** — solve `ex1223b` normally, capture the interior point via the callback.
* **Phase 2** — create a fresh solver with `ESH.InteriorPoint.Strategy = OnlyExternal`,
  inject the captured point, and verify the same optimal solution is reached.

In [ ]:
import SHOTpy

# ── Helper: build ex1223b ─────────────────────────────────────────────────
def build_ex1223b_nb(env):
    problem = SHOTpy.Problem(env)
    problem.name = "ex1223b"
    x1 = SHOTpy.Variable("x1", 0, SHOTpy.VariableType.Real,   0.0, 10.0)
    x2 = SHOTpy.Variable("x2", 1, SHOTpy.VariableType.Real,   0.0, 10.0)
    x3 = SHOTpy.Variable("x3", 2, SHOTpy.VariableType.Real,   0.0, 10.0)
    b4 = SHOTpy.Variable("b4", 3, SHOTpy.VariableType.Binary, 0.0, 1.0)
    b5 = SHOTpy.Variable("b5", 4, SHOTpy.VariableType.Binary, 0.0, 1.0)
    b6 = SHOTpy.Variable("b6", 5, SHOTpy.VariableType.Binary, 0.0, 1.0)
    b7 = SHOTpy.Variable("b7", 6, SHOTpy.VariableType.Binary, 0.0, 1.0)
    for v in [x1, x2, x3, b4, b5, b6, b7]:
        problem.addVariable(v)
    obj_expr = ((b4 - 1.0)**2 + (b5 - 2.0)**2 + (b6 - 1.0)**2
                - SHOTpy.log(1.0 + b7)
                + (x1 - 1.0)**2 + (x2 - 2.0)**2 + (x3 - 3.0)**2)
    problem.setObjective(SHOTpy.NonlinearObjectiveFunction(
        SHOTpy.ObjectiveDirection.Minimize, obj_expr, 0.0))
    lt1 = SHOTpy.LinearTerms()
    for v in [x1, x2, x3, b4, b5, b6]:
        lt1.add(SHOTpy.LinearTerm(1.0, v))
    problem.addConstraint(SHOTpy.LinearConstraint(0, "e1", lt1, SHOTpy.SHOT_DBL_MIN, 5.0))
    problem.addConstraint(SHOTpy.NonlinearConstraint(1, "e2",
        b6**2 + x1**2 + x2**2 + x3**2, SHOTpy.SHOT_DBL_MIN, 5.5))
    for i, (v1, v2, rhs) in enumerate([(x1,b4,1.2),(x2,b5,1.8),(x3,b6,2.5),(x1,b7,1.2)], 2):
        lt = SHOTpy.LinearTerms()
        lt.add(SHOTpy.LinearTerm(1.0, v1)); lt.add(SHOTpy.LinearTerm(1.0, v2))
        problem.addConstraint(SHOTpy.LinearConstraint(i, f"e{i+1}", lt, SHOTpy.SHOT_DBL_MIN, rhs))
    problem.addConstraint(SHOTpy.NonlinearConstraint(6, "e7",
        b5**2 + x2**2, SHOTpy.SHOT_DBL_MIN, 1.64))
    problem.addConstraint(SHOTpy.NonlinearConstraint(7, "e8",
        b6**2 + x3**2, SHOTpy.SHOT_DBL_MIN, 4.25))
    problem.addConstraint(SHOTpy.NonlinearConstraint(8, "e9",
        b5**2 + x3**2, SHOTpy.SHOT_DBL_MIN, 4.64))
    problem.finalize()
    return problem

# ═══════════════════════════════════════════════════════════════════════════
# Phase 1 — Solve normally, capture the interior point via callback
# ═══════════════════════════════════════════════════════════════════════════
print("Phase 1: solving ex1223b with default ESH strategy")
solver1 = SHOTpy.Solver()
solver1.updateSetting("Output.Console.LogLevel", 2)
solver1.updateSetting("Model.Convexity.AssumeConvex", True)
env1 = solver1.getEnvironment()
solver1.setProblem(build_ex1223b_nb(env1))

captured_points = []

def capture_cb(data):
    """Fired after internal strategy finds interior point(s); we just record them."""
    captured_points.extend(data.currentInteriorPoints)
    print(f"  [Phase 1 callback] {len(data.currentInteriorPoints)} interior point(s) found internally")
    return None  # keep them unchanged

solver1.registerCallback(SHOTpy.EventType.ExternalESHRootsearchPointsSelection, capture_cb)
solver1.solveProblem()

phase1_obj = solver1.getPrimalSolution().objValue
print(f"Phase 1 optimal objective: {phase1_obj:.6f}  (expected ≈ 4.5796)")
print(f"Captured {len(captured_points)} interior point(s) to use in Phase 2")

# ═══════════════════════════════════════════════════════════════════════════
# Phase 2 — Skip internal NLP; inject the captured point via callback
# ═══════════════════════════════════════════════════════════════════════════
print("\nPhase 2: solving with ESH.InteriorPoint.Strategy = OnlyExternal")
solver2 = SHOTpy.Solver()
solver2.updateSetting("Output.Console.LogLevel", 2)
solver2.updateSetting("Model.Convexity.AssumeConvex", True)
solver2.updateSetting("Dual.ESH.InteriorPoint.Strategy", 1)  # OnlyExternal
env2 = solver2.getEnvironment()
solver2.setProblem(build_ex1223b_nb(env2))

def inject_cb(data):
    """Provide the captured Phase-1 interior point; internal NLP was skipped."""
    print(f"  [Phase 2 callback] currentInteriorPoints={len(data.currentInteriorPoints)} "
          f"(should be 0 for OnlyExternal)")
    return captured_points  # inject the captured points

solver2.registerCallback(SHOTpy.EventType.ExternalESHRootsearchPointsSelection, inject_cb)
solver2.solveProblem()

phase2_obj = solver2.getPrimalSolution().objValue
print(f"Phase 2 optimal objective: {phase2_obj:.6f}")
print(f"Objectives match: {abs(phase2_obj - phase1_obj) < 1e-4}")

### ESH Interior Point via Auxiliary Problem

When no interior point is available from a previous solve — for example at the very start
of a workflow — you can find one by solving a small auxiliary NLP.

**Idea:** take the original problem, relax all integer/binary variables to continuous, add an
auxiliary slack variable $\mu$, subtract $\mu$ from every nonlinear constraint, and minimise
$\mu$.  If the optimal $\mu^\star < 0$ the solution point is strictly interior to all
nonlinear constraints, and can be fed directly into the
`ExternalESHRootsearchPointsSelection` callback.

```
Modified problem (auxiliary)
──────────────────────────────────────────────────────────────────────
min  μ
s.t.  g_i(x) − μ  ≤  b_i   for each nonlinear constraint i
      all original linear constraints
      binary variables relaxed to Real [0, 1]
      μ ∈ [−100, 100]
──────────────────────────────────────────────────────────────────────
```

The cell below applies this to `ex1223b`:

1. **Auxiliary solve** — builds the modified problem, solves it, and extracts the 7-variable
   interior point by dropping $\mu$ (the last variable).
2. **Main solve** — re-solves the original `ex1223b` with `ESH.InteriorPoint.Strategy =
   OnlyExternal`, injecting the auxiliary-problem interior point through the callback.

In [ ]:
import SHOTpy

# ── Helper reused from above ─────────────────────────────────────────────────
# build_ex1223b_nb(env) is defined in the earlier ESH cell; reuse it here.

# ══════════════════════════════════════════════════════════════════════════════
# Phase 1 — Auxiliary problem: minimise mu to find a strictly interior point
# ══════════════════════════════════════════════════════════════════════════════
print("Phase 1: solving auxiliary minimize-mu problem to find an interior point")

solver_aux = SHOTpy.Solver()
solver_aux.updateSetting("Output.Console.LogLevel", 6)  # silent
solver_aux.updateSetting("Model.Reformulation.Quadratics.Strategy", 0)  # Nonlinear
env_aux = solver_aux.getEnvironment()

problem_aux = SHOTpy.Problem(env_aux)
problem_aux.name = "ex1223b_interior"

# Original 7 variables — binary b4-b7 relaxed to Real [0, 1]
x1 = SHOTpy.Variable("x1", 0, SHOTpy.VariableType.Real, 0.0, 10.0)
x2 = SHOTpy.Variable("x2", 1, SHOTpy.VariableType.Real, 0.0, 10.0)
x3 = SHOTpy.Variable("x3", 2, SHOTpy.VariableType.Real, 0.0, 10.0)
b4 = SHOTpy.Variable("b4", 3, SHOTpy.VariableType.Real, 0.0, 1.0)
b5 = SHOTpy.Variable("b5", 4, SHOTpy.VariableType.Real, 0.0, 1.0)
b6 = SHOTpy.Variable("b6", 5, SHOTpy.VariableType.Real, 0.0, 1.0)
b7 = SHOTpy.Variable("b7", 6, SHOTpy.VariableType.Real, 0.0, 1.0)
# Auxiliary slack variable
mu = SHOTpy.Variable("mu", 7, SHOTpy.VariableType.Real, -100.0, 100.0)
for v in [x1, x2, x3, b4, b5, b6, b7, mu]:
    problem_aux.addVariable(v)

# Objective: minimize mu
lt_obj = SHOTpy.LinearTerms()
lt_obj.add(SHOTpy.LinearTerm(1.0, mu))
problem_aux.setObjective(
    SHOTpy.LinearObjectiveFunction(SHOTpy.ObjectiveDirection.Minimize, lt_obj, 0.0)
)

# e1: linear, unchanged
lt1 = SHOTpy.LinearTerms()
for v in [x1, x2, x3, b4, b5, b6]:
    lt1.add(SHOTpy.LinearTerm(1.0, v))
problem_aux.addConstraint(SHOTpy.LinearConstraint(0, "e1", lt1, SHOTpy.SHOT_DBL_MIN, 5.0))

# e2, e7, e8, e9: quadratic constraints with -mu subtracted
problem_aux.addConstraint(SHOTpy.NonlinearConstraint(
    1, "e2", b6**2 + x1**2 + x2**2 + x3**2 - mu, SHOTpy.SHOT_DBL_MIN, 5.5))

# e3-e6: linear, unchanged
for idx, (va, vb, rhs, nm) in enumerate(
        [(x1, b4, 1.2, "e3"), (x2, b5, 1.8, "e4"),
         (x3, b6, 2.5, "e5"), (x1, b7, 1.2, "e6")], start=2):
    lt = SHOTpy.LinearTerms()
    lt.add(SHOTpy.LinearTerm(1.0, va))
    lt.add(SHOTpy.LinearTerm(1.0, vb))
    problem_aux.addConstraint(SHOTpy.LinearConstraint(idx, nm, lt, SHOTpy.SHOT_DBL_MIN, rhs))

problem_aux.addConstraint(SHOTpy.NonlinearConstraint(
    6, "e7", b5**2 + x2**2 - mu, SHOTpy.SHOT_DBL_MIN, 1.64))
problem_aux.addConstraint(SHOTpy.NonlinearConstraint(
    7, "e8", b6**2 + x3**2 - mu, SHOTpy.SHOT_DBL_MIN, 4.25))
problem_aux.addConstraint(SHOTpy.NonlinearConstraint(
    8, "e9", b5**2 + x3**2 - mu, SHOTpy.SHOT_DBL_MIN, 4.64))

problem_aux.finalize()
solver_aux.setProblem(problem_aux)
solver_aux.solveProblem()

aux_sol      = solver_aux.getPrimalSolution()
optimal_mu   = aux_sol.objValue
interior_pt  = list(aux_sol.point[:7])   # drop mu (index 7)

print(f"Optimal mu : {optimal_mu:.6f}")
if optimal_mu < 0:
    print(f"  mu < 0  → strictly interior to all nonlinear constraints ✓")
else:
    print(f"  Warning: mu ≥ 0 — point may not be strictly interior")
print(f"Interior point (7 vars): {[round(v, 4) for v in interior_pt]}")

# ══════════════════════════════════════════════════════════════════════════════
# Phase 2 — Solve original ex1223b with OnlyExternal, injecting the aux point
# ══════════════════════════════════════════════════════════════════════════════
print("\nPhase 2: solving ex1223b with ESH.InteriorPoint.Strategy = OnlyExternal")
print("         injecting the auxiliary-problem interior point via callback")

solver2 = SHOTpy.Solver()
solver2.updateSetting("Output.Console.LogLevel", 6)
solver2.updateSetting("Model.Reformulation.Quadratics.Strategy", 0)  # Nonlinear
solver2.updateSetting("Dual.ESH.InteriorPoint.Strategy", 1)          # OnlyExternal
env2 = solver2.getEnvironment()
solver2.setProblem(build_ex1223b_nb(env2))

def inject_aux_point(data):
    print(f"  [Callback] fired — currentInteriorPoints={len(data.currentInteriorPoints)} "
          f"(should be 0 for OnlyExternal)")
    return [interior_pt]   # one interior point

solver2.registerCallback(SHOTpy.EventType.ExternalESHRootsearchPointsSelection, inject_aux_point)
solver2.solveProblem()

phase2_obj = solver2.getPrimalSolution().objValue
print(f"\nPhase 2 objective  : {phase2_obj:.6f}  (known optimum ≈ 4.5796)")
print(f"Objectives match   : {abs(phase2_obj - 4.579582) < 0.01}")

## Summary

This tutorial covered:

1. **Basic setup** - Creating environments and problems
2. **Variables** - Real, integer, and binary variables
3. **Linear programming** - LP formulation and solving
4. **Mixed-integer programming** - MIP problems
5. **Quadratic programming** - QP and QCQP problems
6. **Nonlinear programming** - Using exp, log, sin, cos, and other functions
7. **File I/O** - Reading problems from OSiL and GAMS files
8. **Settings** - Configuring solver parameters
9. **Advanced features** - Signomial terms, solution details, and problem properties
10. **Callbacks** - Reacting to solver events, early termination, and custom hyperplane injection
11. **External bounds** - Injecting dual bounds and primal solution candidates to accelerate convergence
12. **ESH interior point callback** - Injecting or modifying interior points for the ESH algorithm
13. **ESH interior point via auxiliary problem** - Finding a strictly interior point by minimising a slack variable μ

### Additional Resources

- **SHOT Documentation**: https://shotsolver.dev
- **SHOT Repository**: https://github.com/coin-or/SHOT
- **Test Files**: See the `test/python/` directory for more examples

### Tips for Best Results

1. Always call `problem.finalize()` before creating a solver
2. Use appropriate variable bounds to help the solver
3. Configure MIP and NLP subsolvers based on problem characteristics
4. Adjust termination criteria (time limit, gap tolerances) as needed
5. Check solution status and bounds after solving